<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle%20Study%20Practice/07_House_Prices_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 구글 드라이브 연결 (로그인 팝업이 뜨면 확인만 눌러주세요)
from google.colab import drive
drive.mount('/content/drive')

# 2. 구글 드라이브에 저장해둔 열쇠(access_token)를 코랩 보안 폴더로 자동 복사
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

# 3. 캐글 API로 MNIST 데이터 초고속 다운로드 및 압축 해제
!pip install -q kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

In [ ]:
import os

for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd

train_df = pd.read_csv(os.path.join(path, 'train.csv'))
test_df = pd.read_csv(os.path.join(path, 'test.csv'))
# print(train_df.head())


print(len(train_df), len(test_df))

mean_price = train_df['SalePrice'].mean()
print(f"훈련 데이터의 평균 집값: ${mean_price:,.2f}")

print(test_df[['Id', 'GrLivArea', 'TotRmsAbvGrd']].head())

In [ ]:
price_per_sqft = train_df['SalePrice'].sum() / train_df['GrLivArea'].sum()
print(f"면적 1단위당 계산된 가격: ${price_per_sqft:.2f}")

predictions = []
for area in test_df['GrLivArea']:
    predicted_price = area * price_per_sqft
    predictions.append(predicted_price)

print(len(predictions))

In [ ]:
submission_data = {
    'Id': test_df['Id'],
    'SalePrice': predictions
}

submission_df = pd.DataFrame(submission_data)

submission_df.to_csv('house_price_submission.csv', index=False)

In [ ]:
train_set = pd.read_csv(os.path.join(path, 'train.csv'))

print(type(train_set))

print(train_set.shape)
print(train_set.columns.tolist())
print(train_set.dtypes)

In [ ]:
import numpy as np

missing = train_set.isnull().sum()
missing = missing[missing > 0]
print("Features with missing values:")
print(missing.sort_values(ascending=False))

cat_cols = train_set.select_dtypes(include=['object']).columns

print("\nCategorical features:")
print(cat_cols.tolist())

num_cols = train_set.select_dtypes(include=['int64', 'float64']).columns

print(f"type of num_cols: {type(num_cols)}")
print("\nNumerical features:")
print(num_cols.tolist())

skewness = train_set[num_cols].skew().sort_values(ascending=False)

print("\nMost skewed numerical features:")
print(skewness.head(10))

In [ ]:
data = {
    '이름': ['A', 'B', 'C'],
    '나이': [25, np.nan, 30],       # np.nan은 결측치를 의미
    '직업': ['학생', '회사원', None]   # None도 결측치로 인식
}

df = pd.DataFrame(data)

print(type(df))
print(df.isnull())
print(df.isnull().sum())

# '나이' 컬럼이 결측치인 행만 필터링
null_age_rows = df[df['나이'].isnull()]
print(null_age_rows)

`isnull()`에 대해 궁금해서 구글링하고, 코드를 따라 쳐보았다.

`null_age_rows = df[df['나이'].isnull()]` 처럼 '나이' 컬럼이 결측치인 행만 필터링할 수도 있다.

type도 궁금해서 `type(train_set), type(df)` 둘 다 찍어봤지만 다음과 같이 동일했다.
`<class 'pandas.core.frame.DataFrame'>`

In [ ]:
import matplotlib.pyplot as plt

features = ["SalePrice", "OverallQual", "GrLivArea", "GarageCars"]

for feature in features:
    plt.figure(figsize=(6, 4))
    plt.hist(train_set[feature], bins=30)
    plt.title(f"Histogram of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.show()

`plt.hist()`에서 bins는 데이터를 나누는 구간(막대)의 개수나 범위를 설정하는 매개변수

In [ ]:
import seaborn as sns

corr = train_set[num_cols].corr()

plt.figure(figsize=(12, 10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
target_corr = corr["SalePrice"].sort_values(ascending=False)
print(target_corr.head(15))

In [ ]:
# 궁금해서 적은 코드
# 상관관계가 0.7 이상인 변수 쌍만 리스트로 보기

# 1. 상삼각행렬을 이용해 대칭되는 중복 데이터와 자기 자신(대각선)을 마스킹
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
high_corr_matrix = corr.where(mask)

# 2. 0.7보다 큰 값만 필터링한 후, 1차원 시리즈로 변환(unstack)
high_corr_pairs = high_corr_matrix[high_corr_matrix > 0.7].unstack()

# 3. 결측치(NaN) 제거 및 값 정렬
high_corr_pairs = high_corr_pairs.dropna().sort_values(ascending=False)

print("---상관관계가 0.7 이상인 변수 쌍---")
print(high_corr_pairs)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

X_train = train_set.drop('SalePrice', axis=1)
y_train = train_set['SalePrice']

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBRegressor(random_state=0))
])

scores = -cross_val_score(
    pipeline,
    X_train,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("CV RMSE:", scores.mean())